# Article 메달리온 테이블 구성

기존 `data_platform/workspace/notebooks/pipelines/price_medallion/medallion.ipynb`의 테이블 설계를 현재 k3d 환경으로 옮긴 노트북입니다.

## Goal

`lakehouse` Catalog에 Bronze·Silver·Gold·Ops 테이블 4개를 생성하고 컬럼·필수값·파티션·저장 속성을 확인합니다.
업무 데이터 수집·변환·LLM 호출은 실행하지 않습니다.

## Setup

- 환경: k3d `dev`, Namespace `finance-platform`, Spark 4.1.2 / Iceberg 1.11.0.
- Catalog: Iceberg REST → PostgreSQL, 데이터: Garage의 `s3://warehouse/`.
- 인증은 Pod의 기존 Secret 환경변수를 사용하며 노트북에 저장하지 않습니다.
- UTC 기준. `sql/` 디렉터리를 이 노트북과 함께 유지하세요.
- 기존 테이블이 있으면 먼저 스키마를 검사합니다. 불일치 시 중단하며 자동 삭제·교체하지 않습니다.
- 원본의 Bronze `ar_published_at`은 최초 생성문에 통합했습니다. 기존 데이터의 언어 변경 UPDATE는 실행하지 않습니다.
- `ops.article_pipeline_state`는 원본에 생성 정의가 없어 만들지 않습니다. 이 노트북에서는 존재 여부만 확인합니다.


In [1]:
from pathlib import Path
import re
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("article-medallion-tables").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")
SQL_DIR = Path.cwd() / "sql"
assert SQL_DIR.is_dir(), f"SQL 디렉터리가 없습니다: {SQL_DIR}"
CATALOG = "lakehouse"


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/18 13:51:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Steps

### 1. 테이블 계약과 기존 테이블 확인

모든 테이블은 Iceberg v2 + Parquet + Zstd입니다. Iceberg에는 PK 제약을 만들지 않으며 기사·버전의 중복 제어는 파이프라인 MERGE 계약으로 처리합니다.


In [2]:
# 컬럼 순서, Spark 타입, nullable 여부를 명시합니다.
CONTRACTS = {
    "bronze_article": {
        "table": "lakehouse.bronze.article",
        "columns": [
            ("ingest_key", "string", False), ("job_id", "string", False),
            ("start_url", "string", False), ("source_url", "string", False),
            ("canonical_url", "string", True), ("ar_title", "string", False),
            ("ar_content", "string", False), ("metadata_json", "string", False),
            ("meta_description", "string", True), ("language", "string", True),
            ("http_status_code", "int", True), ("collected_at", "timestamp", False),
            ("ingested_at", "timestamp", False), ("ar_published_at", "timestamp", True),
        ],
        "partition": "days(collected_at)",
        "extra_properties": {},
    },
    "silver_article": {
        "table": "lakehouse.silver.article",
        "columns": [
            ("article_id", "string", False), ("au_url", "string", False),
            ("ar_title", "string", False), ("ar_content", "string", False),
            ("language", "string", True), ("dt_created_at", "timestamp", False),
            ("dt_ingested_at", "timestamp", False),
        ],
        "partition": "bucket(32,article_id),days(dt_created_at)",
        "extra_properties": {"write.merge.mode": "copy-on-write", "write.distribution-mode": "hash"},
    },
    "gold_graph_article": {
        "table": "lakehouse.gold.graph_article",
        "columns": [
            ("article_id", "string", False), ("au_url", "string", False),
            ("ar_title", "string", False), ("ar_content", "string", False),
            ("language", "string", True), ("dt_created_at", "timestamp", False),
            ("dt_ingested_at", "timestamp", False), ("ont_version", "string", False),
            ("ont_analyzed_at", "timestamp", False), ("ont_candidate_json", "string", False),
        ],
        "partition": "days(ont_analyzed_at)",
        "extra_properties": {},
    },
    "ops_ontology_batch_job": {
        "table": "lakehouse.ops.ontology_batch_job",
        "columns": [
            ("batch_key", "string", False), ("google_job_name", "string", True),
            ("ont_version", "string", False), ("article_ids_json", "string", False),
            ("status", "string", False), ("next_retry_at", "timestamp", True),
            ("dt_created_at", "timestamp", False), ("dt_submitted_at", "timestamp", True),
            ("dt_completed_at", "timestamp", True),
        ],
        "partition": "days(dt_created_at)",
        "extra_properties": {},
    },
}

def validate_table(contract):
    table = contract["table"]
    actual = [(f.name, f.dataType.simpleString(), f.nullable) for f in spark.table(table).schema.fields]
    if actual != contract["columns"]:
        raise RuntimeError(f"{table}: 컬럼/타입/nullability 불일치. 자동 변경하지 않습니다.")
    ddl = "\n".join(str(r[0]) for r in spark.sql(f"SHOW CREATE TABLE {table}").collect())
    compact = re.sub(r"[`\s]+", "", ddl.lower())
    assert "usingiceberg" in compact, f"{table}: Iceberg가 아닙니다."
    assert f"partitionedby({contract['partition']})" in compact, f"{table}: 파티션 불일치"
    props = {r[0]: r[1] for r in spark.sql(f"SHOW TBLPROPERTIES {table}").collect()}
    required = {"format-version": "2", "write.format.default": "parquet", "write.parquet.compression-codec": "zstd", **contract["extra_properties"]}
    for key, value in required.items():
        assert props.get(key) == value, f"{table}: {key} 불일치"
    assert f"location's3://warehouse/" in compact, f"{table}: warehouse 위치 불일치"
    return len(actual)

# 전체 기존 테이블을 먼저 검사한 뒤 생성 단계로 진행합니다.
for contract in CONTRACTS.values():
    if spark.catalog.tableExists(contract["table"]):
        validate_table(contract)
        print(contract["table"], "기존 테이블 계약 일치")
    else:
        print(contract["table"], "신규 생성 대상")


lakehouse.bronze.article 기존 테이블 계약 일치
lakehouse.silver.article 기존 테이블 계약 일치
lakehouse.gold.graph_article 기존 테이블 계약 일치
lakehouse.ops.ontology_batch_job 기존 테이블 계약 일치


### 2. 네임스페이스 생성


In [3]:
for namespace in ("bronze", "silver", "gold", "ops"):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{namespace}")
spark.sql("SHOW NAMESPACES IN lakehouse").show(truncate=False)


+--------------+
|namespace     |
+--------------+
|bronze        |
|gold          |
|ops           |
|silver        |
|platform_smoke|
+--------------+



### Bronze: 수집 원본 이력


In [4]:
contract = CONTRACTS["bronze_article"]
spark.sql((SQL_DIR / "bronze_article.sql").read_text())
column_count = validate_table(contract)
print(contract["table"], "검증 완료, 컬럼", column_count, "개")
spark.sql(f"DESCRIBE TABLE {contract['table']}").show(30, truncate=False)


lakehouse.bronze.article 검증 완료, 컬럼 14 개
+----------------+------------------+-----------------------------+
|col_name        |data_type         |comment                      |
+----------------+------------------+-----------------------------+
|ingest_key      |string            |수집 작업 내 중복 판별 키    |
|job_id          |string            |기사 수집 작업 ID            |
|start_url       |string            |크롤링 시작 URL              |
|source_url      |string            |수집한 기사 URL              |
|canonical_url   |string            |정규화에 사용할 canonical URL|
|ar_title        |string            |수집한 기사 제목             |
|ar_content      |string            |수집한 기사 본문             |
|metadata_json   |string            |가변 기사 메타데이터 JSON    |
|meta_description|string            |HTML meta description        |
|language        |string            |기사 언어 코드               |
|http_status_code|int               |수집 시점의 HTTP 상태 코드   |
|collected_at    |timestamp         |기사 수집 시각               |
|ingested_at     

### Silver: 기사별 최신 상태


In [5]:
contract = CONTRACTS["silver_article"]
spark.sql((SQL_DIR / "silver_article.sql").read_text())
column_count = validate_table(contract)
print(contract["table"], "검증 완료, 컬럼", column_count, "개")
spark.sql(f"DESCRIBE TABLE {contract['table']}").show(30, truncate=False)


lakehouse.silver.article 검증 완료, 컬럼 7 개
+--------------+----------------------+-------------------------------+
|col_name      |data_type             |comment                        |
+--------------+----------------------+-------------------------------+
|article_id    |string                |canonical URL 해시 기반 기사 ID|
|au_url        |string                |정규화된 기사 URL              |
|ar_title      |string                |정규화된 기사 제목             |
|ar_content    |string                |정규화된 기사 본문             |
|language      |string                |정규화된 기사 언어 코드        |
|dt_created_at |timestamp             |기사 작성 또는 발행 시각       |
|dt_ingested_at|timestamp             |Bronze 적재 시각               |
|              |                      |                               |
|# Partitioning|                      |                               |
|Part 0        |bucket(32, article_id)|                               |
|Part 1        |days(dt_created_at)   |                               |
+----

### Gold: 검증된 온톨로지 결과


In [6]:
contract = CONTRACTS["gold_graph_article"]
spark.sql((SQL_DIR / "gold_graph_article.sql").read_text())
column_count = validate_table(contract)
print(contract["table"], "검증 완료, 컬럼", column_count, "개")
spark.sql(f"DESCRIBE TABLE {contract['table']}").show(30, truncate=False)


lakehouse.gold.graph_article 검증 완료, 컬럼 10 개
+------------------+---------------------+-------------------------------------------------+
|col_name          |data_type            |comment                                          |
+------------------+---------------------+-------------------------------------------------+
|article_id        |string               |기사 식별자                                      |
|au_url            |string               |실제 기사 URL                                    |
|ar_title          |string               |정규화된 기사 제목                               |
|ar_content        |string               |정규화된 기사 본문                               |
|language          |string               |정규화된 기사 언어 코드                          |
|dt_created_at     |timestamp            |기사 최초 생성 시각                              |
|dt_ingested_at    |timestamp            |기사 수집 시각                                   |
|ont_version       |string               |적용한 온톨로지 분석 버전                   

### Ops: 비동기 분석 배치 상태


In [7]:
contract = CONTRACTS["ops_ontology_batch_job"]
spark.sql((SQL_DIR / "ops_ontology_batch_job.sql").read_text())
column_count = validate_table(contract)
print(contract["table"], "검증 완료, 컬럼", column_count, "개")
spark.sql(f"DESCRIBE TABLE {contract['table']}").show(30, truncate=False)


lakehouse.ops.ontology_batch_job 검증 완료, 컬럼 9 개
+----------------+-------------------+--------------------------------+
|col_name        |data_type          |comment                         |
+----------------+-------------------+--------------------------------+
|batch_key       |string             |비동기 분석 배치 식별 키        |
|google_job_name |string             |외부 Batch 작업 이름            |
|ont_version     |string             |적용한 온톨로지 버전            |
|article_ids_json|string             |배치에 포함된 기사 manifest JSON|
|status          |string             |배치 처리 상태                  |
|next_retry_at   |timestamp          |다음 재시도 가능 시각           |
|dt_created_at   |timestamp          |배치 상태 생성 시각             |
|dt_submitted_at |timestamp          |외부 Batch 제출 시각            |
|dt_completed_at |timestamp          |배치 완료 시각                  |
|                |                   |                                |
|# Partitioning  |                   |                                |
|Part 0      

## Checks

테이블 행 수와 참조만 있던 운영 상태 테이블의 존재 여부를 확인합니다.
초기 구성에서는 업무 데이터가 없어 네 테이블의 행 수가 0입니다.
테이블이 커진 환경에서는 전체 COUNT 비용을 고려해 아래 집계를 선택적으로 실행하세요.


In [8]:
summary = []
for contract in CONTRACTS.values():
    table = contract["table"]
    count = spark.table(table).count()
    summary.append((table, len(contract["columns"]), count))
spark.createDataFrame(summary, "table_name string, column_count int, row_count long").show(truncate=False)
state_exists = spark.catalog.tableExists("lakehouse.ops.article_pipeline_state")
print("ops.article_pipeline_state 존재:", state_exists)
print("ARTICLE_TABLES_VALIDATED", len(summary))


+--------------------------------+------------+---------+
|table_name                      |column_count|row_count|
+--------------------------------+------------+---------+
|lakehouse.bronze.article        |14          |0        |
|lakehouse.silver.article        |7           |0        |
|lakehouse.gold.graph_article    |10          |0        |
|lakehouse.ops.ontology_batch_job|9           |0        |
+--------------------------------+------------+---------+

ops.article_pipeline_state 존재: False
ARTICLE_TABLES_VALIDATED 4


## Next Steps

기사 수집기는 Bronze에 append하고, 후속 파이프라인은 Silver 최신 상태와 Gold 온톨로지 버전별 결과를 관리합니다.
이 노트북은 테이블만 준비하며 파이프라인·DAG·LLM 분석을 시작하지 않습니다.
`ops.article_pipeline_state`는 실제 생산자와 스키마 계약을 확정한 후 별도 구성합니다.

아래 셀은 Spark 자원을 반환합니다. 다시 실행하려면 Setup 셀부터 실행하세요.


In [9]:
spark.stop()
